<a href="https://colab.research.google.com/github/elyatlc/extra_task_2/blob/main/ungraded_notebook_week_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10 — Data Aggregation and Group Operations

In data analysis you constantly need to answer questions like:
- *"What is the average tip per day of the week?"*
- *"How many sales did each region make this quarter?"*
- *"What is the spread of scores within each test group?"*

All of these follow the same three-step pattern that Hadley Wickham called **split → apply → combine**:

| Step | What happens |
|------|-------------|
| **Split** | Divide the data into groups based on one or more keys |
| **Apply** | Run a function on each group independently |
| **Combine** | Glue the per-group results back into a single object |

pandas implements this pattern through `groupby`. This notebook walks through each part step by step.

**Sections covered:**
1. How to Think About Group Operations (`groupby` basics)
2. Data Aggregation (`.agg()`)
3. Apply — General Split-Apply-Combine (`.apply()`)
4. Group Transforms (`.transform()`)
5. Pivot Tables and Cross-Tabulation


## Setup

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)  # so random numbers are the same every run

---
## 10.1 How to Think About Group Operations

### The dataset

We'll start with a small, easy-to-read DataFrame so every step is transparent.  
It has two grouping columns (`key1`, `key2`) and two data columns (`data1`, `data2`).  
Notice that some values are `None`/`NaN` — this will matter later.


In [ ]:
df = pd.DataFrame({
    "key1" : ["a", "a", None, "b", "b", "a", None],
    "key2" : pd.Series([1, 2, 1, 2, 1, None, 1], dtype="Int64"),
    "data1": np.random.standard_normal(7),
    "data2": np.random.standard_normal(7),
})
df

---
### Step 1 — Grouping by a single column

The simplest case: split the rows using the values in **one** column.

**Syntax:** `df.groupby("column_name")`

Calling `.groupby()` alone does **not** compute anything.  
It creates a `GroupBy` object — a recipe that knows *how* to split the data,  
but waits for you to tell it *what* to do with each group.


In [ ]:
# This just creates the GroupBy object — no output yet
grouped = df.groupby("key1")
grouped

To actually compute something, call an aggregation method on the GroupBy object.  
Let's ask: *"What is the mean of every numeric column for each value of key1?"*


In [ ]:
# .mean() triggers the computation and returns the result
df.groupby("key1").mean(numeric_only=True)

**Reading the result:**
- The row index is now made up of the unique values from `key1` (`"a"` and `"b"`).
- Each number is the mean of that column for that group.
- Rows where `key1` was `None` are silently excluded (more on this below).

> ⚠️ **Non-numeric columns are automatically dropped** from the aggregation result.  
> This is called a *nuisance column* exclusion. If a column cannot be meaningfully  
> averaged (e.g. a column of strings), pandas drops it rather than raising an error.  
> Always pass `numeric_only=True` explicitly to make your intention clear.


#### Selecting a single column to aggregate

If you only need one column's summary, select it **before** calling the aggregation.  
This is cleaner and faster than aggregating everything then discarding columns.

**Syntax:** `df.groupby("key")["column"].mean()`


In [ ]:
# Full data1 column before grouping — 7 individual values
df["data1"]

In [ ]:
# Mean of data1 only, one value per key1 group
df.groupby("key1")["data1"].mean()

---
### Counting rows in each group: `.size()` and `.count()`

Two methods are often confused — they answer slightly different questions:

| Method | Question answered | NaN treatment |
|--------|-------------------|--------------|
| `.size()` | How many **rows** are in each group? | Counts all rows, including those with NaN *data* values |
| `.count()` | How many **non-null values** are in each column per group? | Skips NaN values |


In [ ]:
# Reminder of the DataFrame
df

In [ ]:
# size — counts every row that belongs to each group
df.groupby("key1").size()

In [ ]:
# count — counts only non-null values, reported per column
# Notice key2 shows 2 for group 'a' because row 5 has key2=NaN
df.groupby("key1").count()

---
### Handling missing group keys: `dropna=False`

By default, any row whose **grouping key** is `None` or `NaN` is simply ignored.  
Look at our DataFrame — rows 2 and 6 have `key1 = None`.  
They don't appear in any group in the results above.

If you want those rows to form their own `NaN` group, pass `dropna=False`:


In [ ]:
# Default behaviour — rows with None key1 are dropped
df.groupby("key1").size()

In [ ]:
# dropna=False — None keys become their own group (shown as NaN)
df.groupby("key1", dropna=False).size()

> ⚠️ This is easy to miss: you may be unknowingly losing rows every time  
> you `groupby` a column that contains nulls. Always check with `.groupby(..., dropna=False).size()`  
> if you suspect missing keys in your data.


---
### Step 2 — Grouping by multiple columns

You can pass a **list of column names** to create groups based on the combination of values.  
Each unique combination becomes its own group.

**Syntax:** `df.groupby(["col1", "col2"])`


In [ ]:
# The data again for reference
df

In [ ]:
# Groups are now every unique (key1, key2) pair
df.groupby(["key1", "key2"]).mean(numeric_only=True)

**Reading the result:**  
The index is now a **MultiIndex** (hierarchical index) with two levels:  
- Outer level: values from `key1`
- Inner level: values from `key2`

Each row shows the mean for that specific combination.  
You can call `.unstack()` to reshape the inner level into columns:


In [ ]:
# unstack() pivots the innermost index level into columns — easier to read
df.groupby(["key1", "key2"])["data1"].mean().unstack()

---
### Peeking inside groups: iterating over a GroupBy object

The GroupBy object is **iterable**. Each iteration yields a `(group_name, group_data)` tuple.  
This is useful for inspecting or debugging what each group actually contains.


In [ ]:
# Each iteration gives: (group name, subset DataFrame)
for name, group_df in df.groupby("key1"):
    print(f"--- Group: {name!r} ---")
    print(group_df)
    print()

With multiple keys, the group name becomes a **tuple**:


In [ ]:
for (k1, k2), group_df in df.groupby(["key1", "key2"]):
    print(f"key1={k1!r}, key2={k2}  →  {len(group_df)} row(s)")

---
## 10.2 Data Aggregation

An **aggregation** reduces a group of values down to a single summary value  
(e.g. the mean of 50 numbers → one number).

pandas has many built-in aggregations, all available as simple string names:

| String | What it computes |
|--------|----------------|
| `"mean"` | Average of non-null values |
| `"sum"` | Sum of non-null values |
| `"min"` / `"max"` | Smallest / largest value |
| `"count"` | Number of non-null values |
| `"std"` / `"var"` | Sample standard deviation / variance |
| `"median"` | Arithmetic median |
| `"first"` / `"last"` | First / last non-null value |
| `"size"` | Total number of rows (including nulls) |

For these examples we'll use the classic **tipping dataset**.


In [ ]:
tips = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
)
# Add a tip percentage column for later examples
tips["tip_pct"] = tips["tip"] / tips["total_bill"]
tips.head()

---
### Aggregating a single column with one function

**Syntax:** `df.groupby("key")["column"].agg("function_name")`

You can also call the function directly as a method: `.mean()`, `.sum()`, etc.  
Both are equivalent — `.agg("mean")` and `.mean()` produce the same result.


In [ ]:
# The full tip_pct column — all 244 individual values
tips["tip_pct"].head(10)

In [ ]:
# After grouping by day: one mean per day (4 values instead of 244)
tips.groupby("day")["tip_pct"].agg("mean")

---
### Applying multiple functions at once

Pass a **list** of function names (or callables) to get all of them in one go.  
The result is a DataFrame where each column corresponds to one function.

**Syntax:** `grouped["col"].agg(["func1", "func2", ...])`


In [ ]:
# Define a custom function — range of values in a group
def peak_to_peak(arr):
    return arr.max() - arr.min()

# Before: group sizes — how many tip_pct values are in each group
tips.groupby(["day", "smoker"])["tip_pct"].size()

In [ ]:
# After: three summary statistics computed in one call
tips.groupby(["day", "smoker"])["tip_pct"].agg(["mean", "std", peak_to_peak])

**What changed:** Instead of 244 individual values we now have a compact summary  
table with one row per (day, smoker) combination and three descriptive statistics.

> ⚠️ **Custom functions are slower** than built-in string aliases like `"mean"` or `"std"`.  
> pandas uses optimised internal paths for built-in functions.  
> Use string aliases whenever possible; only write custom functions when truly needed.


#### Renaming aggregation columns

Lambda functions and custom functions sometimes produce unhelpful column names  
(e.g. `"<lambda_0>"`). Use **(name, function) tuples** to assign clean names:

**Syntax:** `grouped["col"].agg([("new_name", "func"), ...])`


In [ ]:
tips.groupby(["day", "smoker"])["tip_pct"].agg([
    ("average",  "mean"),
    ("spread",   peak_to_peak),
])

---
### Applying different functions to different columns

Pass a **dictionary** mapping column names to their aggregation function(s).  
This lets you aggregate several columns simultaneously, each with its own function.

**Syntax:** `grouped.agg({"col1": "func1", "col2": "func2"})`


In [ ]:
# Before: what the raw tip and size columns look like
tips[["tip", "size"]].head()

In [ ]:
# After: max tip AND total party size — different functions per column
tips.groupby(["day", "smoker"]).agg({
    "tip":  "max",
    "size": "sum",
})

---
### Controlling the result index: `as_index`

By default the group keys become the **row index** of the result.  
This is fine for display but awkward when you want to do further processing  
(e.g. sorting, merging) because you'd need to call `.reset_index()` first.

Pass `as_index=False` to get a plain integer index right away.


In [ ]:
# Default: day and smoker are promoted to the index
tips.groupby(["day", "smoker"]).agg({"tip_pct": "mean"})

In [ ]:
# as_index=False: day and smoker stay as regular columns, integer index added
tips.groupby(["day", "smoker"], as_index=False).agg({"tip_pct": "mean"})

> Both outputs contain identical data — the only difference is where the group keys live  
> (index vs regular columns). Use `as_index=False` when you plan to chain further operations.


---
## 10.3 Apply — General Split-Apply-Combine

`.agg()` is great when you want **one scalar per group**.  
But sometimes you need to return a whole DataFrame or Series per group —  
for example, *"give me the top 3 rows from each group"*.  
That's what `.apply()` is for.  

It is the most flexible GroupBy method: your function can return anything  
(scalar, Series, DataFrame) and pandas will figure out how to combine the pieces.

**Syntax:** `df.groupby("key").apply(my_function, include_groups=False)`


### Example — selecting the top N rows by tip percentage


In [ ]:
# First, define and test the function on the full DataFrame
def top(df, n=5, column="tip_pct"):
    """Return the n rows with the highest values in `column`."""
    return df.sort_values(column, ascending=False)[:n]

# Test on the full dataset — should give the 5 highest tip_pct rows overall
top(tips, n=5)

In [ ]:
# Now apply it per smoker group — top 3 for each group
# include_groups=False: don't pass the grouping column into the function
tips.groupby("smoker").apply(top, n=3, include_groups=False)

**What happened:**  
- pandas split `tips` into two groups: `smoker="No"` and `smoker="Yes"`.  
- `top(group, n=3)` was called on each piece independently.  
- The two results were concatenated, with the group key added as the **outer index level**.  

This outer index level can be removed with `group_keys=False`:


In [ ]:
# group_keys=False: result has just the original row index, no outer group level
tips.groupby("smoker", group_keys=False).apply(top, n=3, include_groups=False)

> ⚠️ **Always pass `include_groups=False`** when your function operates on the whole group  
> DataFrame. Without it, the grouping column is included inside your function, which can  
> cause a `TypeError` (e.g. trying to average a string column) and will produce a  
> `DeprecationWarning` in newer pandas versions.


---
### Practical example — filling missing values with group means

A very common real-world task: fill `NaN` values using the **mean of the same group**  
rather than the overall mean. This preserves group-level patterns.


In [ ]:
states    = ["Ohio", "New York", "Vermont", "Florida",
             "Oregon", "Nevada", "California", "Idaho"]
group_key = ["East"] * 4 + ["West"] * 4

data = pd.Series(np.random.standard_normal(8), index=states)
data[["Vermont", "Nevada", "Idaho"]] = np.nan

print("Data BEFORE filling:")
print(data)
print()
print("Group means (this is what will fill the NaN values):")
print(data.groupby(group_key).mean())

In [ ]:
def fill_mean(group):
    """Replace NaN values with the mean of this group."""
    return group.fillna(group.mean())

print("Data AFTER filling with group-specific means:")
data.groupby(group_key).apply(fill_mean)

Vermont (East) was filled with the **East mean**, while Nevada and Idaho (West)  
were filled with the **West mean** — not the overall mean.  
This is more appropriate when groups have different baseline levels.


---
## 10.4 Group Transforms and "Unwrapped" GroupBys

### `.transform()` — broadcasting group statistics back to every row

`.apply()` can change the **shape** of the output.  
`.transform()` is stricter: it **always returns an object with the same shape as the input**.  

This makes it ideal for adding a new column to your DataFrame that contains  
a group-level statistic aligned to each original row.

**Syntax:** `df.groupby("key")["col"].transform("func")`


In [ ]:
df2 = pd.DataFrame({
    "key":   ["a", "b", "c"] * 4,
    "value": np.arange(12.0),
})
print("Original DataFrame — 12 rows, 3 groups of 4:")
df2

In [ ]:
# .groupby().mean() collapses to ONE value per group — 3 rows total
df2.groupby("key")["value"].mean()

In [ ]:
# .transform("mean") keeps 12 rows — group mean is broadcast to every row in that group
df2.groupby("key")["value"].transform("mean")

Every row in group `"a"` gets `4.5` (mean of 0, 3, 6, 9).  
Every row in group `"b"` gets `5.5`, and so on.  
The output is the same length as the input — it can be added as a new column directly.


In [ ]:
# Typical usage: add the group mean as a new column
df2["group_mean"] = df2.groupby("key")["value"].transform("mean")
df2

---
### Z-score normalisation within groups

A very common transformation: standardise values **relative to their own group**  
(subtract group mean, divide by group standard deviation).  
This is called a *within-group z-score* and removes group-level differences so you  
can compare values across groups on equal footing.


In [ ]:
g = df2.groupby("key")["value"]

print("Values BEFORE normalisation:")
print(df2["value"].values)

In [ ]:
# Subtract each row's group mean, divide by group std
normalized = (df2["value"] - g.transform("mean")) / g.transform("std")

print("Values AFTER within-group z-score normalisation:")
print(normalized.values)
print()
print("Verification — mean per group should be ~0, std should be ~1:")
normalized.groupby(df2["key"]).agg(["mean", "std"]).round(10)

---
### When to use `.apply()` vs `.transform()`

| | `.apply()` | `.transform()` |
|---|---|---|
| **Output shape** | Flexible — can be anything | Must match the input shape |
| **Typical use** | Selecting rows, complex per-group logic | Adding group stats as new columns |
| **Returns per group** | Scalar, Series, or DataFrame | Series of the same length as the group |

> ⚠️ If your transform function accidentally returns fewer rows than the group has,  
> pandas will raise a `ValueError`. Switch to `.apply()` in that case.


---
## 10.5 Pivot Tables and Cross-Tabulation

### What is a pivot table?

A **pivot table** is a 2D summary of your data:  
- Rows correspond to one set of group keys.  
- Columns correspond to another set of group keys.  
- Each cell holds an aggregated value (mean by default).  

Think of it as a more readable alternative to a `groupby` with two keys.  

**Syntax:** `df.pivot_table(index=..., columns=..., values=..., aggfunc=...)`


In [ ]:
# Our tipping dataset
tips.head()

#### Basic pivot table — mean of all numeric columns


In [ ]:
# Equivalent to: tips.groupby(["day","smoker"]).mean(numeric_only=True)
tips.pivot_table(index=["day", "smoker"], values=["total_bill","tip","size","tip_pct"])

#### Choosing specific values and reorganising rows vs columns

Use `values=` to pick which columns to summarise,  
`index=` for what goes on the rows, and `columns=` for what goes across the top.


In [ ]:
# Before pivot_table: flat structure — every combination is its own row
tips.groupby(["time", "day", "smoker"], observed=True)[["tip_pct", "size"]].mean().head(8)

# When you set observed=True, the group operation only shows and calculates
#results for category values that actually exist in your current data.
# Unused categories are ignored


In [ ]:
# After pivot_table: smoker spreads across columns — much easier to compare
tips.pivot_table(
    index   = ["time", "day"],
    columns = "smoker",
    values  = ["tip_pct", "size"],
    observed= True
)

#### Adding row and column totals with `margins=True`

`margins=True` adds an `"All"` row and column containing the overall aggregation  
(ignoring the group split), useful for sanity-checking your subtotals.


In [ ]:
tips.pivot_table(
    index   = ["time", "day"],
    columns = "smoker",
    values  = ["tip_pct", "size"],
    margins = True,
    observed= True
)

#### Filling empty cells with `fill_value`

Some (row, column) combinations may not exist in the data, leaving `NaN` in the table.  
Use `fill_value=` to replace those with a sensible default (e.g. `0`).


In [ ]:
# Without fill_value — NaN appears where no data exists for that combination
tips.pivot_table(
    index   = ["time", "day"],
    columns = "smoker",
    values  = "tip_pct",
    observed= True
)

In [ ]:
# With fill_value=0 — NaN cells are replaced by 0
tips.pivot_table(
    index      = ["time", "day"],
    columns    = "smoker",
    values     = "tip_pct",
    fill_value = 0,
    observed   = True
)

> ⚠️ Be careful with `fill_value=0` — think about whether 0 is meaningful in your context.  
> For tip percentages, 0 is a real possible value. It may be clearer to leave `NaN`  
> and handle missing combinations explicitly downstream.


---
### `pd.crosstab` — counting co-occurrences

`crosstab` is a special-purpose tool that counts **how often two categorical  
variables appear together**. It is equivalent to `pivot_table` with `aggfunc=len`,  
but with a cleaner syntax for the frequency-counting use case.

**Syntax:** `pd.crosstab(rows, columns, margins=True/False)`


In [ ]:
from io import StringIO

raw_data = (
    "Sample Nationality Handedness\n"
    "1 USA Right-handed\n"
    "2 Japan Left-handed\n"
    "3 USA Right-handed\n"
    "4 Japan Right-handed\n"
    "5 Japan Left-handed\n"
    "6 Japan Right-handed\n"
    "7 USA Right-handed\n"
    "8 USA Left-handed\n"
    "9 Japan Right-handed\n"
    "10 USA Right-handed"
)

survey = pd.read_table(StringIO(raw_data), sep=r"\s+")
survey

In [ ]:
# Count how many people from each nationality have each handedness
pd.crosstab(survey["Nationality"], survey["Handedness"], margins=True)

The `All` row/column shows the marginal totals — 5 Japanese, 5 Americans, 3 left-handed, 7 right-handed.  

You can also pass **lists of Series** for multi-level rows or columns:


In [ ]:
# Multi-level rows: combination of time and day
pd.crosstab([tips["time"], tips["day"]], tips["smoker"], margins=True)

> ⚠️ **`crosstab` vs `pivot_table`:**  
> - `crosstab` → always counts rows (frequency table)  
> - `pivot_table` → any aggregation function (mean, sum, max, …)  
>  
> If you need proportions instead of counts, pass `normalize=True` to `crosstab`:


In [ ]:
# normalize="index" → each row sums to 1.0 (shows proportions within each nationality)
pd.crosstab(survey["Nationality"], survey["Handedness"], normalize="index")

---
## Quick Reference

### `groupby` basics
| What you want | Syntax |
|---|---|
| Group by one column | `df.groupby("col")` |
| Group by multiple columns | `df.groupby(["col1", "col2"])` |
| Include NaN keys as a group | `df.groupby("col", dropna=False)` |
| Flat integer index in result | `df.groupby("col", as_index=False)` |
| Count rows per group | `df.groupby("col").size()` |
| Count non-null values per group | `df.groupby("col").count()` |
| Select one column to aggregate | `df.groupby("col")["data"].mean()` |

### Aggregation with `.agg()`
| What you want | Syntax |
|---|---|
| One built-in function | `.agg("mean")` or `.mean()` |
| Several functions | `.agg(["mean", "std", "max"])` |
| Custom column names | `.agg([("avg", "mean"), ("rng", peak_to_peak)])` |
| Different function per column | `.agg({"col1": "mean", "col2": "sum"})` |

### `.apply()` vs `.transform()`
| | `.apply(func, include_groups=False)` | `.transform(func)` |
|---|---|---|
| Output size | Flexible | Same as input |
| Returns | Anything | Series same length as group |
| Use for | Filtering rows, complex logic | Adding group stats as columns |

### Pivot tables & crosstab
| Argument | Purpose |
|---|---|
| `index` | Row labels |
| `columns` | Column labels |
| `values` | What to aggregate |
| `aggfunc` | How to aggregate (default: `"mean"`) |
| `margins=True` | Add row/column totals |
| `fill_value=0` | Replace NaN cells |
| `pd.crosstab(r, c)` | Frequency count of two categorical variables |
| `pd.crosstab(r, c, normalize="index")` | Row-wise proportions |


## Quick Practice (2–3 minutes)

### Task 1
What does this code do?

```python
df.groupby("key1").size()
```

---

### Task 2
Fill in the blank:

```python
df.________("key1").mean()
```

---

### Task 3
True or False?

`dropna=False` keeps missing values as a separate group.

---

### Task 4
Write one line of code to calculate the mean of `data1` for each `key1` group.

---

### Task 5
What is the difference between:

```python
.size()
```

and

```python
.count()
```
(Answer in one sentence.)
